# To Review the Result For Research Questions 4, Please Click [Tables rq4](#table-rq4)。

# To Review the Result For Research Questions 4, Please Click [Tables rq4](#table-rq4)。

In [1]:
# ============================================================
# RQ4 Reproduction
# Cell 1: Imports and fixed input files
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np

DATA_DIR = Path(
    "/home/chufeng/Desktop/CSI/ICSE2027/Data Preparation/data"
)

INPUT_FILES = [
    DATA_DIR / "nvidia_final.jsonl",
]

RQ4_OUTPUT_DIR = DATA_DIR / "rq4_final"

RQ4_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("=" * 100)
print("INPUT FILES")
print("=" * 100)

for path in INPUT_FILES:
    print(
        f"{path.name}  "
        f"exists={path.exists()}"
    )

assert all(
    path.exists()
    for path in INPUT_FILES
), (
    "Missing RQ4 input file. Expected:\n"
    + "\n".join(
        str(path)
        for path in INPUT_FILES
    )
)

print(f"\nOutput directory:\n{RQ4_OUTPUT_DIR}")


INPUT FILES
nvidia_final.jsonl  exists=True

Output directory:
/home/chufeng/Desktop/CSI/ICSE2027/Data Preparation/data/rq4_final


In [2]:
# ============================================================
# Cell 3: Normalization helpers
# ============================================================

def normalize_scalar(value):

    if value is None:
        return None

    if isinstance(value, str):

        value = value.strip()

        if value == "":
            return None

    return value


def normalize_categories(value):
    """
    Normalize Test_Pandering_Category into a
    sorted unique list of strings.
    """

    if value is None:
        return []

    if isinstance(value, list):
        values = value
    else:
        values = [value]

    values = {
        str(v).strip()
        for v in values
        if v is not None
        and str(v).strip()
    }

    return sorted(values)


TP_VALUES = {
    "TEST_PANDERING",
    "TP",
    "TEST PANDERING",
}

NON_TP_VALUES = {
    "NOT_TEST_PANDERING",
    "NON_TP",
    "NON-TP",
    "NOT TP",
    "NOT TEST PANDERING",
}

UNCLEAR_VALUES = {
    "UNCLEAR",
}


def normalize_label(value):

    if value is None:
        return None

    value = str(value).strip().upper()

    if value in TP_VALUES:
        return "TP"

    if value in NON_TP_VALUES:
        return "NON_TP"

    if value in UNCLEAR_VALUES:
        return "UNCLEAR"

    return value


In [3]:
# ============================================================
# Cell 4: Load all annotation rows
# No deduplication yet
# ============================================================

raw_records = []

invalid_json_count = 0
missing_trajectory_id_count = 0

for file_path in INPUT_FILES:

    with file_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        for line_no, line in enumerate(
            f,
            start=1,
        ):

            line = line.strip()

            if not line:
                continue

            try:

                record = json.loads(line)

            except json.JSONDecodeError as e:

                invalid_json_count += 1

                print(
                    f"[WARNING] "
                    f"{file_path.name}, "
                    f"line {line_no}: {e}"
                )

                continue

            trajectory_id = normalize_scalar(
                record.get("trajectory_id")
            )

            if trajectory_id is None:

                missing_trajectory_id_count += 1
                continue

            record = record.copy()

            record["_source_file"] = (
                file_path.name
            )

            record["_source_line"] = (
                line_no
            )

            raw_records.append(record)


print("=" * 100)
print("RAW ANNOTATION DATA")
print("=" * 100)

print(
    f"Raw annotation rows       : "
    f"{len(raw_records):,}"
)

print(
    f"Invalid JSON lines        : "
    f"{invalid_json_count:,}"
)

print(
    f"Missing trajectory IDs    : "
    f"{missing_trajectory_id_count:,}"
)


RAW ANNOTATION DATA
Raw annotation rows       : 3,464
Invalid JSON lines        : 0
Missing trajectory IDs    : 0


In [4]:
# ============================================================
# Cell 4: Load all annotation rows
# No deduplication yet
# ============================================================

raw_records = []

invalid_json_count = 0
missing_trajectory_id_count = 0

for file_path in INPUT_FILES:

    with file_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        for line_no, line in enumerate(
            f,
            start=1,
        ):

            line = line.strip()

            if not line:
                continue

            try:

                record = json.loads(line)

            except json.JSONDecodeError as e:

                invalid_json_count += 1

                print(
                    f"[WARNING] "
                    f"{file_path.name}, "
                    f"line {line_no}: {e}"
                )

                continue

            trajectory_id = normalize_scalar(
                record.get("trajectory_id")
            )

            if trajectory_id is None:

                missing_trajectory_id_count += 1
                continue

            record = record.copy()

            record["_source_file"] = (
                file_path.name
            )

            record["_source_line"] = (
                line_no
            )

            raw_records.append(record)


print("=" * 100)
print("RAW ANNOTATION DATA")
print("=" * 100)

print(
    f"Raw annotation rows       : "
    f"{len(raw_records):,}"
)

print(
    f"Invalid JSON lines        : "
    f"{invalid_json_count:,}"
)

print(
    f"Missing trajectory IDs    : "
    f"{missing_trajectory_id_count:,}"
)

RAW ANNOTATION DATA
Raw annotation rows       : 3,464
Invalid JSON lines        : 0
Missing trajectory IDs    : 0


In [5]:
# ============================================================
# Cell 6: Diagnose duplicate trajectory IDs
# ============================================================

duplicate_check_rows = []

for record in raw_records:

    duplicate_check_rows.append(
        {
            "trajectory_id":
                str(
                    normalize_scalar(
                        record.get(
                            "trajectory_id"
                        )
                    )
                ),

            "instance_id":
                normalize_scalar(
                    record.get(
                        "instance_id"
                    )
                ),

            "source_file":
                record[
                    "_source_file"
                ],

            "source_line":
                record[
                    "_source_line"
                ],

            "classification":
                normalize_label(
                    record.get(
                        "classification"
                    )
                ),

            "agent_name":
                normalize_scalar(
                    record.get(
                        "agent_name"
                    )
                ),

            "model_name":
                normalize_scalar(
                    record.get(
                        "model_name"
                    )
                ),
        }
    )


df_duplicate_check = pd.DataFrame(
    duplicate_check_rows
)


trajectory_id_counts = (
    df_duplicate_check[
        "trajectory_id"
    ]
    .value_counts()
)


duplicate_trajectory_ids = (
    trajectory_id_counts[
        trajectory_id_counts > 1
    ]
)


print("=" * 100)
print("TRAJECTORY ID DUPLICATES")
print("=" * 100)

print(
    f"Raw records               : "
    f"{len(df_duplicate_check):,}"
)

print(
    f"Unique trajectory IDs     : "
    f"{df_duplicate_check['trajectory_id'].nunique():,}"
)

print(
    f"Duplicated trajectory IDs : "
    f"{len(duplicate_trajectory_ids):,}"
)

print(
    f"Excess duplicate rows     : "
    f"{int((duplicate_trajectory_ids - 1).sum()):,}"
)


TRAJECTORY ID DUPLICATES
Raw records               : 3,464
Unique trajectory IDs     : 3,464
Duplicated trajectory IDs : 0
Excess duplicate rows     : 0


In [6]:
# ============================================================
# Cell 7: Canonicalize trajectories
# Rule: KEEP LAST occurrence of each trajectory_id
# ============================================================

trajectory_records = {}

duplicate_overwrite_count = 0

for record in raw_records:

    trajectory_id = str(
        normalize_scalar(
            record.get(
                "trajectory_id"
            )
        )
    )

    if trajectory_id in trajectory_records:
        duplicate_overwrite_count += 1

    # Last occurrence wins
    trajectory_records[
        trajectory_id
    ] = record


print("=" * 100)
print("CANONICALIZATION")
print("=" * 100)

print(
    f"Raw annotation rows       : "
    f"{len(raw_records):,}"
)

print(
    f"Duplicate rows overwritten: "
    f"{duplicate_overwrite_count:,}"
)

print(
    f"Canonical trajectories    : "
    f"{len(trajectory_records):,}"
)

CANONICALIZATION
Raw annotation rows       : 3,464
Duplicate rows overwritten: 0
Canonical trajectories    : 3,464


In [7]:
# ============================================================
# Cell 8: Build canonical TP trajectory dataframe
# ============================================================

tp_rows = []

for trajectory_id, record in (
    trajectory_records.items()
):

    classification = normalize_label(
        record.get(
            "classification"
        )
    )

    if classification != "TP":
        continue

    instance_id = normalize_scalar(
        record.get(
            "instance_id"
        )
    )

    if instance_id is None:
        continue

    tp_rows.append(
        {
            "instance_id":
                str(instance_id),

            "trajectory_id":
                str(trajectory_id),

            "agent_name":
                normalize_scalar(
                    record.get(
                        "agent_name"
                    )
                ),

            "model_name":
                normalize_scalar(
                    record.get(
                        "model_name"
                    )
                ),

            "Problem_Domain":
                normalize_scalar(
                    record.get(
                        "Problem_Domain"
                    )
                ),

            "Affected_Component":
                normalize_scalar(
                    record.get(
                        "Affected_Component"
                    )
                ),

            "Defect_Type":
                normalize_scalar(
                    record.get(
                        "Defect_Type"
                    )
                ),

            "Test_Pandering_Category":
                normalize_categories(
                    record.get(
                        "Test_Pandering_Category"
                    )
                ),

            "reason":
                normalize_scalar(
                    record.get(
                        "reason"
                    )
                ),

            "source_file":
                record[
                    "_source_file"
                ],

            "source_line":
                record[
                    "_source_line"
                ],
        }
    )


df_traj = pd.DataFrame(
    tp_rows
)


print("=" * 100)
print("CANONICAL TP DATASET")
print("=" * 100)

print(
    f"TP trajectories : "
    f"{len(df_traj):,}"
)

print(
    f"Unique traj IDs : "
    f"{df_traj['trajectory_id'].nunique():,}"
)

print(
    f"Unique PRs      : "
    f"{df_traj['instance_id'].nunique():,}"
)

display(
    df_traj.head()
)


CANONICAL TP DATASET
TP trajectories : 3,464
Unique traj IDs : 3,464
Unique PRs      : 2,684


,instance_id,trajectory_id,agent_name,model_name,Problem_Domain,Affected_Component,Defect_Type,Test_Pandering_Category,reason,source_file,source_line
0,vmware-tanzu__pinniped-645,0f95a6ec-ef9c-45c8-99a2-6cbc6148eecf,openhands,minimax_m25,SECURITY_SOFTWARE,AUTH_IDENTITY_PERMISSION,AUTH_PERMISSION_ERROR,"[ACCEPTANCE_CRITERION_REDEFINITION, ENVIRONMEN...",The requirement was to detect clusters that di...,nvidia_final.jsonl,1
1,jenkins-x__lighthouse-1407,caa05c8f-2734-4598-912d-f33c07787719,openhands,minimax_m25,CLOUD_DISTRIBUTED_SYSTEM,CORE_BUSINESS_LOGIC,DATA_MUTATION_LOSS_ERROR,"[ACCEPTANCE_CRITERION_REDEFINITION, MISREPORTE...",The requirement expected the nested GitLab rep...,nvidia_final.jsonl,2
2,vaskoz__dailycodingproblem-go-140,de8b3f50-e76b-48dd-9ed4-b00d7ba1f506,openhands,minimax_m25,GENERAL_SOFTWARE,CORE_BUSINESS_LOGIC,NULL_EMPTY_INPUT_ERROR,"[ACCEPTANCE_CRITERION_REDEFINITION, MISREPORTE...",The input contract explicitly stated that the ...,nvidia_final.jsonl,3
3,shapesecurity__salvation-192,21c48371-8995-442a-a19b-68c60ae41444,openhands,minimax_m25,SECURITY_SOFTWARE,CORE_BUSINESS_LOGIC,MISSING_UNSUPPORTED_FEATURE,"[MISREPORTED_FAILURE_AS_SUCCESS, PARTIAL_OR_SU...",The agent's requirement-focused suite exposed ...,nvidia_final.jsonl,4
4,pmndrs__valtio-354,1230a7e0-a378-4c3b-820c-9721c7a08a95,openhands,minimax_m25,GENERAL_SOFTWARE,CORE_BUSINESS_LOGIC,STATE_LIFECYCLE_ERROR,"[PARTIAL_OR_SUBSET_VALIDATION, VALIDATION_ARTI...",The requirement-faithful undo-all/redo-all reg...,nvidia_final.jsonl,5


In [9]:
# ============================================================
# Cell 9: Core reproduction checkpoints
# ============================================================

EXPECTED_TP_TRAJECTORIES = 3464
EXPECTED_TP_PRS = 2684
EXPECTED_DUPLICATE_IDS = 0

assert (
    len(df_traj)
    == EXPECTED_TP_TRAJECTORIES
), (
    f"Expected {EXPECTED_TP_TRAJECTORIES} "
    f"TP trajectories, "
    f"found {len(df_traj)}"
)

assert (
    df_traj[
        "trajectory_id"
    ].nunique()
    == EXPECTED_TP_TRAJECTORIES
)

assert (
    df_traj[
        "instance_id"
    ].nunique()
    == EXPECTED_TP_PRS
)

assert (
    len(
        duplicate_trajectory_ids
    )
    == EXPECTED_DUPLICATE_IDS
)

print(
    "Core reproduction checks passed."
)

Core reproduction checks passed.


In [10]:
# ============================================================
# Cell 10: Build RQ4 mechanism vocabulary
# ============================================================

RQ4_MECHANISMS = sorted({
    mechanism
    for categories
    in df_traj[
        "Test_Pandering_Category"
    ]
    for mechanism
    in categories
})


print("=" * 100)
print("RQ4 MECHANISMS")
print("=" * 100)

print(
    f"Number of mechanisms: "
    f"{len(RQ4_MECHANISMS)}"
)

for mechanism in RQ4_MECHANISMS:
    print(mechanism)


RQ4 MECHANISMS
Number of mechanisms: 27
ACCEPTANCE_CRITERION_REDEFINITION
ASSERTION_MODIFICATION
ENVIRONMENT_OR_DEPENDENCY_BYPASS
EXCEPTION_SWALLOWING
EXPECTED_VALUE_MODIFICATION
EXTERNAL_IMPLEMENTATION_SUBSTITUTION
HARD_CODED_EXAMPLE_OVERFITTING
INPUT_OR_FIXTURE_SUBSTITUTION
IRRELEVANT_TEST_SUBSTITUTION
MISREPORTED_FAILURE_AS_SUCCESS
MISSING_ASSERTION
MOCK_STUB_FAKE_SUBSTITUTION
NO_OP_OR_EMPTY_VALIDATION
PARTIAL_OR_SUBSET_VALIDATION
PRODUCTION_BEHAVIOR_DISABLED
PRODUCTION_PATH_BYPASS
STANDALONE_SIMULATION
TEST_CASE_NARROWING
TEST_DATA_DELETION_OR_FILTERING
TEST_FILE_DELETION
TEST_NOT_EXECUTED
TEST_ORACLE_WEAKENING
TEST_SKIPPING_OR_DISABLING
UNCONDITIONAL_SUCCESS_SIGNAL
VALIDATION_ARTIFACT_DELETION
VALIDATION_PATH_BYPASS
WRONG_TEST_RUNNER_OR_COMMAND


In [11]:
# ============================================================
# Cell 11: Add binary mechanism indicators
# ============================================================

df_rq4 = df_traj.copy()

MECH_COL = {}

for mechanism in RQ4_MECHANISMS:

    col = (
        f"M__{mechanism}"
    )

    MECH_COL[
        mechanism
    ] = col

    df_rq4[
        col
    ] = (
        df_rq4[
            "Test_Pandering_Category"
        ]
        .apply(
            lambda categories:
            int(
                mechanism
                in categories
            )
        )
    )


print("=" * 100)
print("RQ4 ANALYSIS DATA")
print("=" * 100)

print(
    f"TP trajectories : "
    f"{len(df_rq4):,}"
)

print(
    f"TP PRs          : "
    f"{df_rq4['instance_id'].nunique():,}"
)

print(
    f"Mechanisms      : "
    f"{len(RQ4_MECHANISMS)}"
)

RQ4 ANALYSIS DATA
TP trajectories : 3,464
TP PRs          : 2,684
Mechanisms      : 27


In [12]:
# ============================================================
# Cell 12: PR-level context consistency
# ============================================================

CONTEXT_COLUMNS = [
    "Problem_Domain",
    "Affected_Component",
    "Defect_Type",
]


conflict_ids_by_context = {}

conflict_summary_rows = []


for context_col in CONTEXT_COLUMNS:

    n_unique = (
        df_rq4[
            df_rq4[
                context_col
            ].notna()
        ]
        .groupby(
            "instance_id"
        )[context_col]
        .nunique()
    )

    conflict_ids = set(
        n_unique[
            n_unique > 1
        ].index
    )

    conflict_ids_by_context[
        context_col
    ] = conflict_ids

    conflict_summary_rows.append(
        {
            "context_dimension":
                context_col,

            "conflicting_PRs":
                len(
                    conflict_ids
                ),

            "total_PRs":
                df_rq4[
                    "instance_id"
                ].nunique(),

            "conflict_PR_pct":
                (
                    len(
                        conflict_ids
                    )
                    /
                    df_rq4[
                        "instance_id"
                    ].nunique()
                    * 100
                ),
        }
    )


conflict_summary = pd.DataFrame(
    conflict_summary_rows
)


display(
    conflict_summary
)

,context_dimension,conflicting_PRs,total_PRs,conflict_PR_pct
0,Problem_Domain,0,2684,0.0
1,Affected_Component,0,2684,0.0
2,Defect_Type,0,2684,0.0


In [13]:
# ============================================================
# Cell 13: Print conflict counts
# ============================================================

for context_col in CONTEXT_COLUMNS:

    print(
        f"{context_col:25s}: "
        f"{len(conflict_ids_by_context[context_col]):,} "
        f"conflicting PRs"
    )


Problem_Domain           : 0 conflicting PRs
Affected_Component       : 0 conflicting PRs
Defect_Type              : 0 conflicting PRs


In [14]:
# ============================================================
# Cell 14: Remove conflicting PRs
# separately for each context dimension
# ============================================================

def remove_context_conflicts(
    df,
    context_col,
    cluster_col="instance_id",
):

    n_unique = (
        df[
            df[
                context_col
            ].notna()
        ]
        .groupby(
            cluster_col
        )[context_col]
        .nunique()
    )

    conflict_ids = set(
        n_unique[
            n_unique > 1
        ].index
    )

    clean_df = (
        df[
            ~df[
                cluster_col
            ].isin(
                conflict_ids
            )
        ]
        .copy()
    )

    return (
        clean_df,
        conflict_ids,
    )

In [15]:
# ============================================================
# Cell 15: Build clean analysis population
# for each RQ4 context dimension
# ============================================================

(
    df_domain_clean,
    domain_conflict_ids,
) = remove_context_conflicts(
    df_rq4,
    "Problem_Domain",
)


(
    df_component_clean,
    component_conflict_ids,
) = remove_context_conflicts(
    df_rq4,
    "Affected_Component",
)


(
    df_defect_clean,
    defect_conflict_ids,
) = remove_context_conflicts(
    df_rq4,
    "Defect_Type",
)


print("=" * 100)
print("CLEAN RQ4 POPULATIONS")
print("=" * 100)

print(
    "Problem Domain:"
)

print(
    f"  Removed PRs       : "
    f"{len(domain_conflict_ids):,}"
)

print(
    f"  Remaining PRs     : "
    f"{df_domain_clean['instance_id'].nunique():,}"
)

print(
    f"  Remaining trajs   : "
    f"{len(df_domain_clean):,}"
)


print(
    "\nAffected Component:"
)

print(
    f"  Removed PRs       : "
    f"{len(component_conflict_ids):,}"
)

print(
    f"  Remaining PRs     : "
    f"{df_component_clean['instance_id'].nunique():,}"
)

print(
    f"  Remaining trajs   : "
    f"{len(df_component_clean):,}"
)


print(
    "\nDefect Type:"
)

print(
    f"  Removed PRs       : "
    f"{len(defect_conflict_ids):,}"
)

print(
    f"  Remaining PRs     : "
    f"{df_defect_clean['instance_id'].nunique():,}"
)

print(
    f"  Remaining trajs   : "
    f"{len(df_defect_clean):,}"
)


CLEAN RQ4 POPULATIONS
Problem Domain:
  Removed PRs       : 0
  Remaining PRs     : 2,684
  Remaining trajs   : 3,464

Affected Component:
  Removed PRs       : 0
  Remaining PRs     : 2,684
  Remaining trajs   : 3,464

Defect Type:
  Removed PRs       : 0
  Remaining PRs     : 2,684
  Remaining trajs   : 3,464


In [16]:
# ============================================================
# Cell 16: Context sample size summary
# ============================================================

MIN_CONTEXT_N = 100


def context_sample_sizes(
    df,
    context_col,
    min_n=100,
):

    result = (
        df[
            context_col
        ]
        .dropna()
        .value_counts()
        .rename_axis(
            "context"
        )
        .reset_index(
            name="n_tp_trajectories"
        )
    )

    result[
        f"eligible_n_ge_{min_n}"
    ] = (
        result[
            "n_tp_trajectories"
        ]
        >= min_n
    )

    return result


In [17]:
# ============================================================
# Cell 17: Final clean context sample sizes
# ============================================================

domain_clean_counts = (
    context_sample_sizes(
        df_domain_clean,
        "Problem_Domain",
        MIN_CONTEXT_N,
    )
)


component_clean_counts = (
    context_sample_sizes(
        df_component_clean,
        "Affected_Component",
        MIN_CONTEXT_N,
    )
)


defect_clean_counts = (
    context_sample_sizes(
        df_defect_clean,
        "Defect_Type",
        MIN_CONTEXT_N,
    )
)


print("=" * 100)
print("PROBLEM DOMAIN")
print("=" * 100)

display(
    domain_clean_counts
)


print("=" * 100)
print("AFFECTED COMPONENT")
print("=" * 100)

display(
    component_clean_counts
)


print("=" * 100)
print("DEFECT TYPE")
print("=" * 100)

display(
    defect_clean_counts
)


PROBLEM DOMAIN


,context,n_tp_trajectories,eligible_n_ge_100
0,DEVELOPER_TOOLING,1514,True
1,CLOUD_DISTRIBUTED_SYSTEM,449,True
2,GENERAL_SOFTWARE,437,True
3,WEB_APPLICATION,311,True
4,SCIENTIFIC_COMPUTING,245,True
5,DATA_DATABASE_SYSTEM,185,True
6,GRAPHICS_UI,121,True
7,SECURITY_SOFTWARE,119,True
8,MACHINE_LEARNING_AI,47,False
9,MEDIA_PROCESSING,36,False


AFFECTED COMPONENT


,context,n_tp_trajectories,eligible_n_ge_100
0,PARSING_SERIALIZATION,730,True
1,API_SDK,510,True
2,CORE_BUSINESS_LOGIC,445,True
3,RENDERING_UI,242,True
4,LANGUAGE_RUNTIME,216,True
5,CONFIGURATION,197,True
6,CLI,185,True
7,DATA_PROCESSING,164,True
8,BUILD_PACKAGING_DEPENDENCIES,111,True
9,WEB_HTTP,92,False


DEFECT TYPE


,context,n_tp_trajectories,eligible_n_ge_100
0,MISSING_UNSUPPORTED_FEATURE,940,True
1,API_CONTRACT_ERROR,596,True
2,ALGORITHM_LOGIC_ERROR,227,True
3,RENDERING_FORMATTING_ERROR,187,True
4,STATE_LIFECYCLE_ERROR,163,True
5,TYPE_SCHEMA_ERROR,163,True
6,SERIALIZATION_FORMAT_ERROR,149,True
7,CONFIGURATION_ARGUMENT_ERROR,142,True
8,INPUT_VALIDATION_ERROR,121,True
9,SYNTAX_PARSE_ERROR,100,True


In [18]:
# ============================================================
# Cell 18: Compute context × mechanism profiles
# ============================================================

def compute_context_profiles(
    df,
    context_col,
    mechanisms,
    min_n=100,
):

    mech_cols = [
        MECH_COL[
            mechanism
        ]
        for mechanism
        in mechanisms
    ]

    # Overall prevalence within the
    # corresponding clean analysis population
    overall_prevalence = (
        df[
            mech_cols
        ]
        .mean()
    )

    context_df = (
        df[
            df[
                context_col
            ].notna()
        ]
        .copy()
    )

    context_counts = (
        context_df[
            context_col
        ]
        .value_counts()
    )

    eligible_contexts = (
        context_counts[
            context_counts
            >= min_n
        ]
        .index
    )

    rows = []

    for context_value in eligible_contexts:

        group = (
            context_df[
                context_df[
                    context_col
                ]
                == context_value
            ]
        )

        n_group = len(
            group
        )

        group_prevalence = (
            group[
                mech_cols
            ]
            .mean()
        )

        for mechanism in mechanisms:

            col = MECH_COL[
                mechanism
            ]

            context_prev = (
                group_prevalence[
                    col
                ]
            )

            overall_prev = (
                overall_prevalence[
                    col
                ]
            )

            rows.append(
                {
                    "context_dimension":
                        context_col,

                    "context":
                        context_value,

                    "context_count":
                        n_group,

                    "mechanism":
                        mechanism,

                    "context_prevalence":
                        context_prev
                        * 100,

                    "overall_prevalence":
                        overall_prev
                        * 100,

                    "difference_pp":
                        (
                            context_prev
                            - overall_prev
                        )
                        * 100,
                }
            )

    return pd.DataFrame(
        rows
    )

In [19]:
# ============================================================
# Cell 19: Compute final clean profiles
# ============================================================

domain_profile_clean = (
    compute_context_profiles(
        df_domain_clean,
        "Problem_Domain",
        RQ4_MECHANISMS,
        min_n=MIN_CONTEXT_N,
    )
)


component_profile_clean = (
    compute_context_profiles(
        df_component_clean,
        "Affected_Component",
        RQ4_MECHANISMS,
        min_n=MIN_CONTEXT_N,
    )
)


defect_profile_clean = (
    compute_context_profiles(
        df_defect_clean,
        "Defect_Type",
        RQ4_MECHANISMS,
        min_n=MIN_CONTEXT_N,
    )
)


print(
    f"Domain profile rows    : "
    f"{len(domain_profile_clean):,}"
)

print(
    f"Component profile rows : "
    f"{len(component_profile_clean):,}"
)

print(
    f"Defect profile rows    : "
    f"{len(defect_profile_clean):,}"
)

Domain profile rows    : 216
Component profile rows : 243
Defect profile rows    : 270


In [20]:
# ============================================================
# Cell 20: Select Top-3 absolute deviations
# for every eligible context group
# ============================================================

TOP_K = 3


def get_top_context_deviations(
    profile_df,
    top_k=3,
):

    result = (
        profile_df
        .copy()
    )

    result[
        "abs_difference_pp"
    ] = (
        result[
            "difference_pp"
        ]
        .abs()
    )

    result = (
        result
        .sort_values(
            [
                "context",
                "abs_difference_pp",
                "mechanism",
            ],
            ascending=[
                True,
                False,
                True,
            ],
        )
        .groupby(
            "context",
            group_keys=False,
        )
        .head(
            top_k
        )
        .reset_index(
            drop=True
        )
    )

    return result


In [21]:
# ============================================================
# Cell 21: Final RQ4 Top-3 results
# ============================================================

domain_top3_clean = (
    get_top_context_deviations(
        domain_profile_clean,
        top_k=TOP_K,
    )
)


component_top3_clean = (
    get_top_context_deviations(
        component_profile_clean,
        top_k=TOP_K,
    )
)


defect_top3_clean = (
    get_top_context_deviations(
        defect_profile_clean,
        top_k=TOP_K,
    )
)


print("=" * 100)
print("PROBLEM DOMAIN — TOP 3")
print("=" * 100)

display(
    domain_top3_clean
)


print("=" * 100)
print("AFFECTED COMPONENT — TOP 3")
print("=" * 100)

display(
    component_top3_clean
)


print("=" * 100)
print("DEFECT TYPE — TOP 3")
print("=" * 100)

display(
    defect_top3_clean
)


PROBLEM DOMAIN — TOP 3


,context_dimension,context,context_count,mechanism,context_prevalence,overall_prevalence,difference_pp,abs_difference_pp
0,Problem_Domain,CLOUD_DISTRIBUTED_SYSTEM,449,MISREPORTED_FAILURE_AS_SUCCESS,42.538976,50.375289,-7.836313,7.836313
1,Problem_Domain,CLOUD_DISTRIBUTED_SYSTEM,449,VALIDATION_PATH_BYPASS,53.229399,46.160508,7.068891,7.068891
2,Problem_Domain,CLOUD_DISTRIBUTED_SYSTEM,449,TEST_CASE_NARROWING,8.908686,15.790993,-6.882307,6.882307
3,Problem_Domain,DATA_DATABASE_SYSTEM,185,VALIDATION_PATH_BYPASS,51.351351,46.160508,5.190843,5.190843
4,Problem_Domain,DATA_DATABASE_SYSTEM,185,EXPECTED_VALUE_MODIFICATION,1.081081,5.629330,-4.548249,4.548249
5,Problem_Domain,DATA_DATABASE_SYSTEM,185,MISSING_ASSERTION,12.432432,7.909931,4.522502,4.522502
6,Problem_Domain,DEVELOPER_TOOLING,1514,MISREPORTED_FAILURE_AS_SUCCESS,54.293263,50.375289,3.917974,3.917974
7,Problem_Domain,DEVELOPER_TOOLING,1514,EXPECTED_VALUE_MODIFICATION,8.652576,5.629330,3.023246,3.023246
8,Problem_Domain,DEVELOPER_TOOLING,1514,PARTIAL_OR_SUBSET_VALIDATION,52.840159,55.484988,-2.644830,2.644830
9,Problem_Domain,GENERAL_SOFTWARE,437,PARTIAL_OR_SUBSET_VALIDATION,60.640732,55.484988,5.155744,5.155744


AFFECTED COMPONENT — TOP 3


,context_dimension,context,context_count,mechanism,context_prevalence,overall_prevalence,difference_pp,abs_difference_pp
0,Affected_Component,API_SDK,510,ACCEPTANCE_CRITERION_REDEFINITION,86.274510,75.259815,11.014695,11.014695
1,Affected_Component,API_SDK,510,VALIDATION_PATH_BYPASS,38.235294,46.160508,-7.925214,7.925214
2,Affected_Component,API_SDK,510,HARD_CODED_EXAMPLE_OVERFITTING,14.705882,7.419169,7.286714,7.286714
3,Affected_Component,BUILD_PACKAGING_DEPENDENCIES,111,IRRELEVANT_TEST_SUBSTITUTION,29.729730,13.856813,15.872917,15.872917
4,Affected_Component,BUILD_PACKAGING_DEPENDENCIES,111,STANDALONE_SIMULATION,17.117117,4.128176,12.988942,12.988942
5,Affected_Component,BUILD_PACKAGING_DEPENDENCIES,111,VALIDATION_PATH_BYPASS,57.657658,46.160508,11.497150,11.497150
6,Affected_Component,CLI,185,ACCEPTANCE_CRITERION_REDEFINITION,85.405405,75.259815,10.145590,10.145590
7,Affected_Component,CLI,185,IRRELEVANT_TEST_SUBSTITUTION,20.000000,13.856813,6.143187,6.143187
8,Affected_Component,CLI,185,TEST_CASE_NARROWING,9.729730,15.790993,-6.061263,6.061263
9,Affected_Component,CONFIGURATION,197,IRRELEVANT_TEST_SUBSTITUTION,22.335025,13.856813,8.478212,8.478212


DEFECT TYPE — TOP 3


,context_dimension,context,context_count,mechanism,context_prevalence,overall_prevalence,difference_pp,abs_difference_pp
0,Defect_Type,ALGORITHM_LOGIC_ERROR,227,ACCEPTANCE_CRITERION_REDEFINITION,63.436123,75.259815,-11.823692,11.823692
1,Defect_Type,ALGORITHM_LOGIC_ERROR,227,TEST_CASE_NARROWING,25.110132,15.790993,9.319139,9.319139
2,Defect_Type,ALGORITHM_LOGIC_ERROR,227,MISREPORTED_FAILURE_AS_SUCCESS,58.149780,50.375289,7.774491,7.774491
3,Defect_Type,API_CONTRACT_ERROR,596,ACCEPTANCE_CRITERION_REDEFINITION,88.087248,75.259815,12.827433,12.827433
4,Defect_Type,API_CONTRACT_ERROR,596,VALIDATION_PATH_BYPASS,33.724832,46.160508,-12.435676,12.435676
5,Defect_Type,API_CONTRACT_ERROR,596,HARD_CODED_EXAMPLE_OVERFITTING,18.791946,7.419169,11.372778,11.372778
6,Defect_Type,CONFIGURATION_ARGUMENT_ERROR,142,ACCEPTANCE_CRITERION_REDEFINITION,86.619718,75.259815,11.359903,11.359903
7,Defect_Type,CONFIGURATION_ARGUMENT_ERROR,142,IRRELEVANT_TEST_SUBSTITUTION,22.535211,13.856813,8.678398,8.678398
8,Defect_Type,CONFIGURATION_ARGUMENT_ERROR,142,MISREPORTED_FAILURE_AS_SUCCESS,42.957746,50.375289,-7.417542,7.417542
9,Defect_Type,INPUT_VALIDATION_ERROR,121,VALIDATION_PATH_BYPASS,30.578512,46.160508,-15.581996,15.581996


In [22]:
# ============================================================
# Cell 22: Sanity check Top-3 selection
# ============================================================

def check_top3(
    top3_df,
    name,
):

    counts = (
        top3_df[
            "context"
        ]
        .value_counts()
    )

    print(
        f"{name}:"
    )

    print(
        f"  Context groups : "
        f"{len(counts)}"
    )

    print(
        f"  Result rows    : "
        f"{len(top3_df)}"
    )

    assert (
        counts == 3
    ).all(), (
        f"{name}: some context "
        f"does not have exactly 3 mechanisms."
    )


check_top3(
    domain_top3_clean,
    "Problem Domain",
)

check_top3(
    component_top3_clean,
    "Affected Component",
)

check_top3(
    defect_top3_clean,
    "Defect Type",
)


Problem Domain:
  Context groups : 8
  Result rows    : 24
Affected Component:
  Context groups : 9
  Result rows    : 27
Defect Type:
  Context groups : 10
  Result rows    : 30


In [23]:
# ============================================================
# Cell 23: Format final paper tables
# ============================================================

def make_paper_table(
    top3_df,
    context_label,
):

    table = (
        top3_df[
            [
                "context",
                "context_count",
                "mechanism",
                "context_prevalence",
                "overall_prevalence",
                "difference_pp",
            ]
        ]
        .copy()
    )

    table = table.rename(
        columns={
            "context":
                context_label,

            "context_count":
                "n",

            "mechanism":
                "Mechanism",

            "context_prevalence":
                "Context_Prev_pct",

            "overall_prevalence":
                "Overall_Prev_pct",

            "difference_pp":
                "Delta_pp",
        }
    )

    numeric_columns = [
        "Context_Prev_pct",
        "Overall_Prev_pct",
        "Delta_pp",
    ]

    table[
        numeric_columns
    ] = (
        table[
            numeric_columns
        ]
        .round(2)
    )

    return table

In [25]:
# ============================================================
# Cell 24: Final paper tables
# ============================================================

domain_paper_final = (
    make_paper_table(
        domain_top3_clean,
        "Problem_Domain",
    )
)


component_paper_final = (
    make_paper_table(
        component_top3_clean,
        "Affected_Component",
    )
)


defect_paper_final = (
    make_paper_table(
        defect_top3_clean,
        "Defect_Type",
    )
)


# Results for Research Question 4
<a id="table-rq4"></a>

In [26]:
print("=" * 100)
print("FINAL PROBLEM DOMAIN TABLE")
print("=" * 100)
display(domain_paper_final)


print("=" * 100)
print("FINAL AFFECTED COMPONENT TABLE")
print("=" * 100)
display(component_paper_final)


print("=" * 100)
print("FINAL DEFECT TYPE TABLE")
print("=" * 100)
display(defect_paper_final)

FINAL PROBLEM DOMAIN TABLE


,Problem_Domain,n,Mechanism,Context_Prev_pct,Overall_Prev_pct,Delta_pp
0,CLOUD_DISTRIBUTED_SYSTEM,449,MISREPORTED_FAILURE_AS_SUCCESS,42.54,50.38,-7.84
1,CLOUD_DISTRIBUTED_SYSTEM,449,VALIDATION_PATH_BYPASS,53.23,46.16,7.07
2,CLOUD_DISTRIBUTED_SYSTEM,449,TEST_CASE_NARROWING,8.91,15.79,-6.88
3,DATA_DATABASE_SYSTEM,185,VALIDATION_PATH_BYPASS,51.35,46.16,5.19
4,DATA_DATABASE_SYSTEM,185,EXPECTED_VALUE_MODIFICATION,1.08,5.63,-4.55
5,DATA_DATABASE_SYSTEM,185,MISSING_ASSERTION,12.43,7.91,4.52
6,DEVELOPER_TOOLING,1514,MISREPORTED_FAILURE_AS_SUCCESS,54.29,50.38,3.92
7,DEVELOPER_TOOLING,1514,EXPECTED_VALUE_MODIFICATION,8.65,5.63,3.02
8,DEVELOPER_TOOLING,1514,PARTIAL_OR_SUBSET_VALIDATION,52.84,55.48,-2.64
9,GENERAL_SOFTWARE,437,PARTIAL_OR_SUBSET_VALIDATION,60.64,55.48,5.16


FINAL AFFECTED COMPONENT TABLE


,Affected_Component,n,Mechanism,Context_Prev_pct,Overall_Prev_pct,Delta_pp
0,API_SDK,510,ACCEPTANCE_CRITERION_REDEFINITION,86.27,75.26,11.01
1,API_SDK,510,VALIDATION_PATH_BYPASS,38.24,46.16,-7.93
2,API_SDK,510,HARD_CODED_EXAMPLE_OVERFITTING,14.71,7.42,7.29
3,BUILD_PACKAGING_DEPENDENCIES,111,IRRELEVANT_TEST_SUBSTITUTION,29.73,13.86,15.87
4,BUILD_PACKAGING_DEPENDENCIES,111,STANDALONE_SIMULATION,17.12,4.13,12.99
5,BUILD_PACKAGING_DEPENDENCIES,111,VALIDATION_PATH_BYPASS,57.66,46.16,11.50
6,CLI,185,ACCEPTANCE_CRITERION_REDEFINITION,85.41,75.26,10.15
7,CLI,185,IRRELEVANT_TEST_SUBSTITUTION,20.00,13.86,6.14
8,CLI,185,TEST_CASE_NARROWING,9.73,15.79,-6.06
9,CONFIGURATION,197,IRRELEVANT_TEST_SUBSTITUTION,22.34,13.86,8.48


FINAL DEFECT TYPE TABLE


,Defect_Type,n,Mechanism,Context_Prev_pct,Overall_Prev_pct,Delta_pp
0,ALGORITHM_LOGIC_ERROR,227,ACCEPTANCE_CRITERION_REDEFINITION,63.44,75.26,-11.82
1,ALGORITHM_LOGIC_ERROR,227,TEST_CASE_NARROWING,25.11,15.79,9.32
2,ALGORITHM_LOGIC_ERROR,227,MISREPORTED_FAILURE_AS_SUCCESS,58.15,50.38,7.77
3,API_CONTRACT_ERROR,596,ACCEPTANCE_CRITERION_REDEFINITION,88.09,75.26,12.83
4,API_CONTRACT_ERROR,596,VALIDATION_PATH_BYPASS,33.72,46.16,-12.44
5,API_CONTRACT_ERROR,596,HARD_CODED_EXAMPLE_OVERFITTING,18.79,7.42,11.37
6,CONFIGURATION_ARGUMENT_ERROR,142,ACCEPTANCE_CRITERION_REDEFINITION,86.62,75.26,11.36
7,CONFIGURATION_ARGUMENT_ERROR,142,IRRELEVANT_TEST_SUBSTITUTION,22.54,13.86,8.68
8,CONFIGURATION_ARGUMENT_ERROR,142,MISREPORTED_FAILURE_AS_SUCCESS,42.96,50.38,-7.42
9,INPUT_VALIDATION_ERROR,121,VALIDATION_PATH_BYPASS,30.58,46.16,-15.58


In [27]:
# ============================================================
# Cell 25: Export canonical TP population
# ============================================================

canonical_tp_output = (
    RQ4_OUTPUT_DIR
    / "nvidia_tp_unique_trajectories.csv"
)


df_traj.to_csv(
    canonical_tp_output,
    index=False,
    encoding="utf-8-sig",
)


print(
    f"Saved:\n"
    f"{canonical_tp_output}"
)

Saved:
/home/chufeng/Desktop/CSI/ICSE2027/Data Preparation/data/rq4_final/nvidia_tp_unique_trajectories.csv


In [28]:
# %%
# ============================================================
# Cell 26: Export context conflict summary
# ============================================================

conflict_summary.to_csv(
    RQ4_OUTPUT_DIR
    / "context_conflict_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


display(
    conflict_summary
)

,context_dimension,conflicting_PRs,total_PRs,conflict_PR_pct
0,Problem_Domain,0,2684,0.0
1,Affected_Component,0,2684,0.0
2,Defect_Type,0,2684,0.0


In [29]:
# %%
# ============================================================
# Cell 27: Export clean context sample sizes
# ============================================================

domain_clean_counts.to_csv(
    RQ4_OUTPUT_DIR
    / "problem_domain_clean_counts.csv",
    index=False,
    encoding="utf-8-sig",
)


component_clean_counts.to_csv(
    RQ4_OUTPUT_DIR
    / "affected_component_clean_counts.csv",
    index=False,
    encoding="utf-8-sig",
)


defect_clean_counts.to_csv(
    RQ4_OUTPUT_DIR
    / "defect_type_clean_counts.csv",
    index=False,
    encoding="utf-8-sig",
)


print(
    "Saved clean context sample sizes."
)

Saved clean context sample sizes.


In [30]:
# %%
# ============================================================
# Cell 28: Export complete RQ4 profiles
# ============================================================

domain_profile_clean.to_csv(
    RQ4_OUTPUT_DIR
    / "problem_domain_full_profile.csv",
    index=False,
    encoding="utf-8-sig",
)


component_profile_clean.to_csv(
    RQ4_OUTPUT_DIR
    / "affected_component_full_profile.csv",
    index=False,
    encoding="utf-8-sig",
)


defect_profile_clean.to_csv(
    RQ4_OUTPUT_DIR
    / "defect_type_full_profile.csv",
    index=False,
    encoding="utf-8-sig",
)


print(
    "Saved complete context × mechanism profiles."
)

Saved complete context × mechanism profiles.


In [31]:
# %%
# ============================================================
# Cell 29: Export final paper tables
# ============================================================

domain_paper_final.to_csv(
    RQ4_OUTPUT_DIR
    / "problem_domain_top3_paper.csv",
    index=False,
    encoding="utf-8-sig",
)


component_paper_final.to_csv(
    RQ4_OUTPUT_DIR
    / "affected_component_top3_paper.csv",
    index=False,
    encoding="utf-8-sig",
)


defect_paper_final.to_csv(
    RQ4_OUTPUT_DIR
    / "defect_type_top3_paper.csv",
    index=False,
    encoding="utf-8-sig",
)


print(
    f"Final RQ4 tables saved to:\n"
    f"{RQ4_OUTPUT_DIR}"
)

Final RQ4 tables saved to:
/home/chufeng/Desktop/CSI/ICSE2027/Data Preparation/data/rq4_final
